# Inference & Advisory Demonstration

### 🎯 Goal of This Notebook
<pre>
The goal of this notebook is to demonstrate how the trained model can predict the disease status
of a single crop leaf image and provide advisory suggestions to farmers.
</pre>
### It ensures:
 - Model can be used on unseen data
 - Predictions map to human-readable disease names
 - Advisory information is returned based on predicted disease
 - Notebook can be used for demo presentations


## Clone the GitHub Repository
### 📌 Purpose
<pre>
 - To obtain the complete project structure locally for exploration and experimentation.
</pre>



In [1]:
# Clone the Repository
!git clone https://github.com/sabin74/Agriculture-Crop-Disease-Detection-Advisory-System.git


Cloning into 'Agriculture-Crop-Disease-Detection-Advisory-System'...
remote: Enumerating objects: 97570, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 97570 (delta 19), reused 47 (delta 18), pack-reused 97473 (from 3)
Receiving objects: 100% (97570/97570), 2.01 GiB | 30.75 MiB/s, done.
Resolving deltas: 100% (609/609), done.
Updating files: 100% (94804/94804), done.


In [2]:
# Set Project Root
import os
os.chdir("/content/Agriculture-Crop-Disease-Detection-Advisory-System")
print("Current Directory: ", os.getcwd())

Current Directory:  /content/Agriculture-Crop-Disease-Detection-Advisory-System


## Import Required Libraries

In [3]:
import json
import time
import random
import numpy as np
from pathlib import Path
from PIL import Image

import tensorflow as tf
from tensorflow import keras
import gradio as gr


# Define Project Paths

In [4]:
BASE_DIR = Path.cwd()

MODEL_PATH = BASE_DIR / "Modeling/models/best_model.keras"
CLASS_INDEX_PATH = BASE_DIR / "Modeling/configs/class_indices.json"
ADVISORY_PATH = BASE_DIR / "Modeling/configs/advisory_system.json"
TEST_DIR = BASE_DIR / "Modeling/Crop Disease Dataset/test"

IMAGE_SIZE = (224, 224)


## Load Model and Metadata

In [5]:
model = keras.models.load_model(MODEL_PATH, compile=False)

with open(CLASS_INDEX_PATH) as f:
    class_indices = json.load(f)
idx_to_class = {v: k for k, v in class_indices.items()}

with open(ADVISORY_PATH, encoding="utf-8") as f:
    advisory_json = json.load(f)

advisory_db = advisory_json["advisory_system"]["diseases"]
general_guidelines = advisory_json["advisory_system"]["general_guidelines"]
disclaimer = advisory_json["advisory_system"]["disclaimer"]


## Image Processing Function

In [6]:
def preprocess_image(img: Image.Image):
    img = img.convert("RGB").resize(IMAGE_SIZE)
    arr = np.array(img) / 255.0
    return np.expand_dims(arr, axis=0)


# Load Sample Image

In [7]:
#  Load 6 random test images
def load_random_test_images(n=12):
    if not TEST_DIR.exists():
        return []

    image_files = []
    for ext in ["*.jpg", "*.jpeg", "*.png"]:
        image_files.extend(TEST_DIR.rglob(ext))

    if len(image_files) == 0:
        return []

    selected = random.sample(image_files, min(n, len(image_files)))
    return [Image.open(p) for p in selected]


sample_images = load_random_test_images()


# Advisory Retrieval Function

In [8]:
def format_advisory(class_name, confidence, language="english"):
    info = advisory_db.get(class_name, {})
    lang = language

    def L(key):
        return info.get(key, {}).get(lang, [])

    crop = info.get(f"crop_{lang}", info.get("crop", "Unknown"))
    disease = info.get(f"disease_{lang}", info.get("disease", "Healthy"))
    severity = info.get("severity", "None")
    emoji = info.get("severity_emoji", "🟢")
    crop_emoji = info.get("crop_emoji", "🌿")

    md = f"""
## {crop_emoji} **{crop} – {disease}**
### {emoji} Severity: **{severity}**
### 📊 Confidence: **{confidence:.2f}%**
---
"""

    if "symptoms" in info:
        md += "### 🧬 Symptoms\n" + "\n".join([f"- {x}" for x in L("symptoms")]) + "\n\n"

    if "causes" in info:
        md += "### 🦠 Causes\n" + "\n".join([f"- {x}" for x in L("causes")]) + "\n\n"

    if "prevention" in info:
        md += "### 🛡️ Prevention\n" + "\n".join([f"- {x}" for x in L("prevention")]) + "\n\n"

    if "treatment_procedure" in info:
        md += "### 💊 Treatment Steps\n" + "\n".join(
            info["treatment_procedure"]["step_by_step"][lang]
        ) + "\n\n"

    if "fungicides" in info:
        md += "### 🧪 Fungicides (Chemical)\n"
        for f in info["fungicides"].get("chemical", []):
            md += f"- **{f['name']}** — {f['dosage']} ({f['application']})\n"

        md += "\n### 🌿 Organic Options\n"
        for f in info["fungicides"].get("organic", []):
            md += f"- **{f['name']}** — {f['recipe']}\n"
        md += "\n"

    if "immediate_actions" in info:
        md += "### ⚠️ Immediate Actions\n" + "\n".join(
            info["immediate_actions"][lang]
        ) + "\n\n"

    md += f"""
### ⏳ Recovery Time
{info.get(f"recovery_time_{lang}", info.get("recovery_time", "N/A"))}

### 📉 Yield Impact
{info.get(f"yield_impact_{lang}", info.get("yield_impact", "N/A"))}

---
### 🛡️ Safety Guidelines
""" + "\n".join(general_guidelines["safety_first"][lang]) + f"""

---
### ⚠️ Disclaimer
_{disclaimer[lang]}_
"""

    return md


## Core Prediction Function

In [9]:
def predict(image, language):
    start = time.time()
    img = preprocess_image(image)
    preds = model.predict(img)[0]

    idx = np.argmax(preds)
    confidence = preds[idx] * 100
    class_name = idx_to_class[idx]

    advisory_md = format_advisory(class_name, confidence, language)
    return advisory_md, round(time.time() - start, 3)


## Interactive Demo Interface (Gradio)

This interface allows:
 - Uploading a leaf image
 - Selecting advisory language
 - Viewing prediction + inference time

In [10]:
# UI Styling

css = """
body {
    background: linear-gradient(135deg, #f0fff4, #e6fffa);
}
.gradio-container {
    font-family: 'Segoe UI', sans-serif;
}
h1, h2 {
    color: #065f46;
}
"""



In [11]:
# Gradio Blocks App
def select_gallery_image(evt: gr.SelectData):
    return evt.value

with gr.Blocks(css=css, theme=gr.themes.Soft(primary_hue="green")) as demo:

    # Header
    gr.Markdown("""
    # 🌱 Smart Crop Disease Detection & Advisory System
    ### AI-powered crop disease diagnosis with step-by-step advisory (🇬🇧 / 🇳🇵)
    """)

    with gr.Row():

        with gr.Column(scale=1):

            image_input = gr.Image(
                type="pil",
                label="📷 Upload Crop Leaf Image",
                height=280
            )

            language = gr.Radio(
                [("🇬🇧 English", "english"), ("🇳🇵 नेपाली", "nepali")],
                value="english",
                label="🌐 Advisory Language"
            )

            analyze_btn = gr.Button("🔍 Analyze Leaf", variant="primary")

            # Gallery BELOW button
            gr.Markdown("### 🖼️ Random Test Samples (2 × 6)")
            gallery = gr.Gallery(
                value=sample_images,
                columns=3,
                rows=2,
                height="auto",
                object_fit="contain",
                show_label=False
            )

        with gr.Column(scale=1.2):

            output_md = gr.Markdown(
                label="🌾 Disease Diagnosis & Advisory"
            )

            inference_time = gr.Number(
                label="⏱️ Inference Time (seconds)",
                precision=3
            )

    # Button action
    analyze_btn.click(
        fn=predict,
        inputs=[image_input, language],
        outputs=[output_md, inference_time]
    )

demo.launch()


/tmp/ipython-input-2744175099.py:5: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=css, theme=gr.themes.Soft(primary_hue="green")) as demo:
/tmp/ipython-input-2744175099.py:5: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css, theme=gr.themes.Soft(primary_hue="green")) as demo:
/usr/local/lib/python3.12/dist-packages/gradio/layouts/column.py:59: UserWarning: 'scale' value should be an integer. Using 1.2 will cause issues.
  warnings.warn(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e403702a7de58cb017.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
